# Module 34 — Exercise 2: Database Sharding Router

Horizontal database sharding divides large datasets across independent database instances based on a shard key.

In this exercise, you will implement a Hash-based Shard Router with cross-shard scatter-gather queries.

| Detail | Value |
|---|---|
| **Time** | 40 minutes |
| **Prerequisites** | Module 34 README |



# Your turn


### Task 1: Implement Hash Shard Router

Implement `ShardRouter(num_shards)`:
- `get_shard_id(key)`: compute `hash(key) % num_shards`.
- `insert(user_id, data)`: route record to appropriate shard.
- `find(user_id)`: directly locate record on its designated shard.
- `scatter_gather_count()`: query all shards and sum total record count.


In [ ]:
# ANSWER 1
import hashlib

class ShardRouter:
    def __init__(self, num_shards: int = 4):
        self.num_shards = num_shards
        self.shards: list[dict[str, dict]] = [{} for _ in range(num_shards)]

    def _shard_id(self, key: str) -> int:
        digest = int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16)
        return digest % self.num_shards

    def insert(self, user_id: str, data: dict) -> int:
        sid = self._shard_id(user_id)
        self.shards[sid][user_id] = data
        return sid

    def find(self, user_id: str) -> dict | None:
        sid = self._shard_id(user_id)
        return self.shards[sid].get(user_id)

    def scatter_gather_count(self) -> int:
        return sum(len(shard) for shard in self.shards)



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

router = ShardRouter(num_shards=4)
for i in range(100):
    router.insert(f"user_{i}", {"age": 20 + i})

found_50 = router.find("user_50")
total_count = router.scatter_gather_count()
shard_sizes = [len(s) for s in router.shards]
print("Shard distribution across 4 shards:", shard_sizes)

results = [
    check(found_50 is not None and found_50["age"] == 70, "Task 1: Direct lookup located record on correct shard"),
    check(total_count == 100, "Task 1: Scatter-gather correctly summed 100 items"),
    check(all(15 <= s <= 35 for s in shard_sizes), "Task 1: Records evenly distributed across shards"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

